# RQ3 — Diagnostic des NaN de la baseline (pli 0, graine 2026)

`adaptation/diag_nan.py` : même réseau, mêmes données, même pli et même graine que la baseline, 1 000 itérations
au plus par variante, journal par itération. A = baseline (fp16, Dice par patch, sans clipping ni warmup),
B = fp32, C = Dice sur le lot, D = unscale_ + clip_grad_norm_(12) + warmup sur 500 itérations.
Au premier poids ou moment non fini : lot, état avant le pas et gradients du pas dans `diag_nan/<variante>/rupture/`.

In [ ]:
# Environnement (baseline : torch 2.11.0+cu130 sur Tesla T4) ; un écart est signalé dans resume.json sans arrêter.
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")

In [ ]:
# Code du projet (dataset privé projet-avc-code) et dépendances absentes de l'image Kaggle.
import hashlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

def find_input(slug):  # dossier d'un dataset attaché, quel que soit le chemin de montage de Kaggle
    for pattern in (slug, f"*/{slug}", f"*/*/{slug}", f"*/*/*/{slug}"):
        found = sorted(Path("/kaggle/input").glob(pattern))
        if found:
            return found[0]
    raise FileNotFoundError(f"Dataset {slug} non attaché : " + str(sorted(Path("/kaggle/input").glob("*"))))

CODE = find_input("projet-avc-code")
REPO = Path("/kaggle/working/PROJET_AVC")
if not REPO.exists():
    archives = sorted(CODE.rglob("projet_avc_code.zip"))
    if archives:
        zipfile.ZipFile(archives[0]).extractall(REPO)
    else:  # Kaggle a décompressé l'archive : copie de l'arborescence
        shutil.copytree(next(CODE.rglob("adaptation/train.py")).parents[1], REPO)
print(json.loads((REPO / "VERSION.json").read_text()))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1", "nibabel==5.4.2",
                "pyarrow"], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Données C1 : liens vers les dossiers de sujets, quelle que soit l'archive d'origine.
INPUT = find_input("atlas-c1-processed")
DATA = Path("/kaggle/working/processed")
DATA.mkdir(exist_ok=True)
sources = [INPUT]
if not any(INPUT.rglob("qc.json")):  # archives restées compressées : extraction sur le disque de travail
    extracted = Path("/kaggle/working/c1_extrait")
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(extracted)
    sources = [extracted]
for qc in (path for source in sources for path in source.rglob("qc.json")):
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
table = DATA / "subjects.parquet"
if not table.exists():
    table.symlink_to(next(INPUT.rglob("subjects.parquet")))
print(sum(1 for _ in DATA.glob("*/qc.json")), "sujets prétraités")
# Même découpage que le code : splits.json du dataset identique à celui du dépôt.
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(next(INPUT.rglob("splits.json"))) == digest(REPO / "splits.json"), "splits.json différent du dépôt"

In [ ]:
# Version du code (VERSION.json du dataset projet-avc-code), contrôlée avant de lancer les variantes.
version = json.loads((REPO / "VERSION.json").read_text())
print("Commit du code :", version["commit"], "-", version["message"])

In [ ]:
# Diagnostic : les quatre variantes l'une après l'autre ; sortie complète aussi dans diag_nan/diag_nan.log.
!mkdir -p /kaggle/working/diag_nan && cd {REPO} && python -u -m adaptation.diag_nan --data-root {DATA} --output /kaggle/working/diag_nan --num-workers 4 2>&1 | tee /kaggle/working/diag_nan/diag_nan.log

In [ ]:
# Fichiers à récupérer (onglet Output).
print((Path("/kaggle/working/diag_nan") / "resume.json").read_text())
!find /kaggle/working/diag_nan -type f -exec ls -la {} +